[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yy726/GPQA-eval/blob/submission-yy726/notebooks/11_sft_grpo_unsloth.ipynb)

# 11 · GRPO on Qwen3 Base（Unsloth + W&B）——从共享 SFT 检查点 S1 开始

**先跑 notebook 15**（共享 SFT 检查点，只训练一次）。本 notebook 默认 `RUN_SFT=False`、`INIT_MODEL="auto"`：自动加载 15 推到 HF 的
`<你的用户名>/qwen3-4b-s1-sft`，只做 GRPO。结构参考 Unsloth 官方 [Qwen3 (4B) GRPO](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Qwen3_(4B)-GRPO.ipynb)。

**X1b 里 GRPO 没用的原因，和这一版怎么改**（`experiments/results.md`）：
1. **起点没有推理可以强化**：X1b 的 SFT 回答是"背一段知识 + 答案"。→ S1 用带分步解答的科学数据训练，在 GPQA 题面下会逐个分析选项（notebook 15）。
2. **题库和 GPQA 不够像**：MMLU-Pro 偏本科、Dr.SCI 是数值题。→ `GRPO_DATA="science"`：一半是 **MegaScience 里留出的 4 选 1 研究生水平科学题**
   （和 GPQA 同一种题面；SFT 没见过它们的解答；只留两份解答答案一致的题），再加 20% MMLU-Pro STEM 和 30% Dr.SCI 数值题。
3. **训练量太小、每步太吵**：X1b 是 100 步 × 4 题。→ 120 步 × 16 题 × 8 个回答 = 1920 道题，学习率 1e-5（Dr.SCI / OLMo 3 全参是 1e-6，LoRA 放大 10 倍）。
4. **量不出差别**：单样本评测的噪声比 GRPO 的效果大。→ 留出验证集（同分布、不参与训练）训练前后各测一次；GPQA 每题采样 8 次，报告配对差值 ± 标准误。

论文里同等规模的参照（Dr.SCI，同一个 Qwen3-4B-Base）：SFT 后再做 RL，GPQA-D 提高 2–3 分；去掉太容易的题再 +2。所以**预期是 1–3 分**，要靠配对比较才看得出来。

**流程（照 OLMo 3：SFT → DPO → RL）**：15（SFT）→ 可选 12（DPO，推到 HF）→ 本 notebook（`INIT_MODEL` = S1 或 DPO 模型）。

**不用守着**：Run all 之后可以离开。adapter、每一段的结果、评测结果都存在 Google Drive（开头会弹一次授权）；断线后重新 Run all 会跳过已完成的评测和 GRPO 阶段；
全部跑完自动推到 HF 私有仓库并断开 runtime。改了配置想重跑某一格，直接重跑即可，不用重启（只有换模型 / 上下文长度才需要 Restart session，会有明确提示）。

| 模式 | 用途 | GRPO | 评测 |
|---|---|---|---|
| `quick` | 验证流程（T4 也行，0.6B；先用 15 的 quick 模式做出 0.6B 的 S1，或设 `RUN_SFT=True`） | 30 步 × 2 题 × 4 | Diamond 前 60 题，1 次采样 |
| `medium` | 1.7B 看变化（要 `RUN_SFT=True` 或自己的 `INIT_MODEL`） | 60 步 × 8 题 × 8 | Diamond，2 次采样 |
| `a100` | **4B 正式实验**（A100 40GB） | 120 步 × 16 题 × 8，两段；预筛 + 训练 + 验证约 2.5 小时（估） | Diamond + Main，≤8192，8 次采样（每次约 17 分钟） |

**生成预算**：排行榜的锁定评测是 8192，本 notebook 默认也用 8192。`GEN_BUDGET=16384` 只在参考分支用，
适合从思维链检查点开始的实验；这时结果里会多一列 `acc_flex_at_8192`（同样的回答按排行榜预算截断后的分数）。

⚠️ 快速评测**不是**排行榜分数（选项顺序与 lm-eval 不同）。正式分数：合并推到 HF 后用 `main` 分支的 `01_eval_gpqa.ipynb`。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 克隆提交分支（数据准备 / 去污染 / 共用代码 scripts/recipe_utils.py 都在里面）
import os, sys
if not os.path.exists("GPQA-eval"):
    !git clone -q -b submission-yy726 https://github.com/yy726/GPQA-eval.git
if os.path.basename(os.getcwd()) != "GPQA-eval":   # 重跑这一格时不要再进一层
    %cd GPQA-eval
!git pull -q
if os.path.abspath("scripts") not in sys.path:
    sys.path.insert(0, os.path.abspath("scripts"))
import recipe_utils
print("reloaded:", recipe_utils.reload_scripts())   # 改了 scripts/ 之后重跑这一格即可，不用重启 runtime

In [ ]:
# 安装（约 3–5 分钟）。输出不隐藏：如果安装报错请看这里。
# 2026-11 实测可用：unsloth 2026.11.3、vllm 0.31、trl 1.15、transformers 5.17（A100）
!pip install --upgrade -q uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install unsloth vllm wandb {_numpy} {_pil}

In [ ]:
import os, importlib.metadata as md, torch
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"   # Unsloth：GRPO 时让 vLLM 和训练共享显存
if torch.cuda.get_device_capability()[0] < 8:   # T4 不支持 FlashInfer 采样器
    os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
    print("T4 detected: FlashInfer disabled for vLLM")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
VERSIONS = {}
for p in ["unsloth", "vllm", "trl", "transformers", "torch", "peft", "wandb"]:
    try: VERSIONS[p] = md.version(p)
    except md.PackageNotFoundError: VERSIONS[p] = None
print(GPU_NAME, GPU_MEM_GB, "GB"); print(VERSIONS)

In [ ]:
# Colab 左侧 🔑 Secrets：HF_TOKEN（必需；先在 https://huggingface.co/datasets/Idavidrein/gpqa 接受条款）
#                      WANDB_API_KEY（强烈建议；没有就不记录 W&B）
for _k in ["HF_TOKEN", "WANDB_API_KEY"]:
    try:
        from google.colab import userdata
        _v = userdata.get(_k)
        if _v:
            os.environ[_k] = _v
    except Exception:
        pass
from huggingface_hub import login, whoami
login(token = os.environ.get("HF_TOKEN"))
HF_USER = whoami(token = os.environ.get("HF_TOKEN"))["name"]
print("HF user:", HF_USER, "|", "W&B key found" if os.environ.get("WANDB_API_KEY") else "No WANDB_API_KEY: W&B logging disabled")

## 0. 配置

In [ ]:
EXP_NAME     = "R1"                           # 实验编号：W&B run 名、报告里用
MODE         = "a100"                         # "quick" | "medium" | "a100"
MODEL_NAME   = "unsloth/Qwen3-4B-Base"        # quick: "unsloth/Qwen3-0.6B-Base"；medium: "unsloth/Qwen3-1.7B-Base"
BASE_MODEL   = "Qwen/Qwen3-4B-Base"           # 提交时填的原始 base
LOAD_IN_4BIT = False
CHAT_STYLE   = "olmo"   # "olmo"（回答以 <|endoftext|> 结束，推荐）| "plain"（无特殊 token）| "chatml"（X1 的旧模板，只作对比）

# 流程开关
RUN_SFT         = False  # False = 从 INIT_MODEL 开始只做 GRPO（推荐）；True = 旧流程：在本 notebook 里先做一遍 X1b 的 SFT
INIT_MODEL      = "auto" # "auto" = notebook 15 的共享 SFT 检查点 f"{HF_USER}/qwen3-4b-s1-sft"；也可以填 notebook 12 的 DPO 模型或本地路径
PUSH_SFT_REPO   = None   # 只在 RUN_SFT=True 时用：SFT 后合并推到 HF 私有仓库
RUN_GRPO        = True
PUSH_FINAL_REPO = "auto" # "auto" = f"{HF_USER}/qwen3-4b-<EXP_NAME>-grpo"（私有；正式评测前在 HF 上改成公开）；None = 只存到持久目录
FINAL_ADAPTER   = None   # None = 合并最后一个阶段；"grpo_lora_s0"（或 RUN_SFT=True 时的 "sft_lora"）= 改为合并持久目录里的这个 adapter
USE_DRIVE       = True   # adapter、阶段结果、评测结果都存到 Google Drive：runtime 重置后不会丢
RESUME          = True   # 已经完成的阶段（评测、每一段 GRPO）直接读结果，不重跑；False = 全部重来
AUTO_DISCONNECT = True   # 全部跑完后自动断开 runtime（不再消耗 CU）。想留着调试就设 False
GEN_BUDGET      = None   # 评测的生成预算。None = 预设（a100: 8192，和排行榜一致）；16384 = 只在参考分支放宽，从思维链检查点（15 的 VARIANT="think"）开始时用
RUN_BASE_EVAL   = False  # base 已测过两次（X1 / X1b：Diamond 26、Main 24，回答不收尾所以要约 30 分钟）；INIT_MODEL 不为空时总是评测起点

# SFT 数据（--include/--exclude 同时匹配来源名和 domain；X1 里 Logic Puzzles 占 29%，现在去掉）
DOLCI_DATASET    = "allenai/Dolci-Instruct-SFT-7B"
DOLCI_INCLUDE    = ["science", "math", "gsm"]
DOLCI_EXCLUDE    = ["safety", "wildguard", "python", "code", "aya", "precise", "flan", "logic"]
THINK_DATASET    = "allenai/Dolci-Think-SFT-7B"
THINK_N          = 0            # >0：再从 Dolci-Think 抽这么多条思维链（长度 ≤ sft_max_tokens）
THINK_INCLUDE    = ["science", "math"]
EXTRA_SFT        = []           # 见 experiments/DATASETS.md，例如 [dict(dataset="MegaScience/TextbookReasoning", n=4000)]
SFT_SELECT       = "coverage"   # "random" | "coverage"（Dr.SCI 的 EESFT：按 4-gram 覆盖度从候选池里挑）
SFT_POOL_FACTOR  = 3            # coverage 时先抽 3 倍候选
COVERAGE_NORM    = "sqrt"       # "none" = 完全照论文（偏向长样本）；"sqrt" = 除以 √长度

# GRPO
GRPO_DATA         = "science"   # "science"（50% MegaScience 留出的 4 选 1 科学题 + 20% MMLU-Pro STEM + 30% Dr.SCI 数值题）| "mix"（X1b：MMLU-Pro + Dr.SCI）| "mmlu_pro_stem" | "drsci" | "sciq"
GRPO_FILTER       = True        # 每段训练前每题采样 4 次，只留答对 1–3 次的（去掉太易 / 太难的题）
CURRICULUM_ROUNDS = 2           # 2 = 两段：前一半较易（Dr.SCI 通过率 0.5–0.875），后一半较难（0.25–0.375），第二段用更新后的模型重新预筛
DRSCI_SUBJECTS    = ["physics", "chemistry", "biology"]
GRPO_LR           = 1e-5        # Dr.SCI / OLMo 3 全参 1e-6，LoRA 放大 10 倍；X1b 的 5e-6 × 100 步没有效果。验证集通过率不动可以试 2e-5
GRPO_MAX_COMPLETION = "auto"    # "auto" = 2 × 模型自己收尾长度的 99 分位（不超过预设上限）；也可以直接填整数
EPSILON_HIGH      = 0.28        # DAPO / OLMo 3 的 clip-higher
FORMAT_WEIGHT     = 0.5         # 格式分（正确性 +2 为主）
OVERLONG_CACHE    = 1024        # 软超长惩罚：最后 1024 token 里从 0 线性扣到 −1

WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "gpqa-hackathon")
WANDB_ENTITY  = os.environ.get("WANDB_ENTITY")

PRESETS = dict(
    quick = dict(max_seq_len=2048, gpu_util=0.7, lora_rank=32,
                 sft_dolci_n=200, sft_mcq_n=200, sft_max_tokens=1024, sft_steps=40, sft_lr=2e-4, sft_batch=4, sft_accum=2,
                 grpo_steps=30, grpo_prompts_per_step=2, num_gen=4, grpo_max_completion=512, filter_k=4, filter_max_tokens=512,
                 val_n=40, eval_configs=["gpqa_diamond"], eval_n=60, eval_max_tokens=1024, eval_samples=1, shuffle_buffer=1000),
    medium = dict(max_seq_len=8192, gpu_util=0.75, lora_rank=32,
                  sft_dolci_n=1000, sft_mcq_n=500, sft_max_tokens=4096, sft_steps=150, sft_lr=2e-4, sft_batch=4, sft_accum=2,
                  grpo_steps=60, grpo_prompts_per_step=8, num_gen=8, grpo_max_completion=2048, filter_k=4, filter_max_tokens=2048,
                  val_n=100, eval_configs=["gpqa_diamond"], eval_n=None, eval_max_tokens=4096, eval_samples=2, shuffle_buffer=5000),
    a100 = dict(max_seq_len=12288, gpu_util=0.8, lora_rank=64,
                sft_dolci_n=20000, sft_mcq_n=2000, sft_max_tokens=6144, sft_steps=-1, sft_lr=1e-4, sft_batch=4, sft_accum=4,
                grpo_steps=120, grpo_prompts_per_step=16, num_gen=8, grpo_max_completion=6144, filter_k=4, filter_max_tokens=3072,
                val_n=400, eval_configs=["gpqa_diamond", "gpqa_main"], eval_n=None, eval_max_tokens=8192, eval_samples=8,
                shuffle_buffer=20000),
)
Q = dict(PRESETS[MODE])
if GEN_BUDGET:   # 放宽预算：上下文 = 预算 + 最长题面；回答长的模型，GRPO 的回答上限和预筛长度也跟着放宽
    Q.update(eval_max_tokens = GEN_BUDGET, max_seq_len = max(Q["max_seq_len"], GEN_BUDGET + 4096))
    if GEN_BUDGET > 8192:
        Q.update(grpo_max_completion = max(Q["grpo_max_completion"], 12288), filter_max_tokens = max(Q["filter_max_tokens"], 8192))
MAX_SEQ_LEN, GPU_UTIL, LORA_RANK = Q["max_seq_len"], Q["gpu_util"], Q["lora_rank"]
from recipe_utils import resolve_init_model, sft_repo_name
INIT_MODEL = None if (RUN_SFT and INIT_MODEL == "auto") else resolve_init_model(INIT_MODEL, HF_USER, MODEL_NAME)
if not RUN_SFT and not INIT_MODEL:
    print("[!] RUN_SFT=False 且没有 INIT_MODEL：会直接在 base 上做 GRPO（base 不会收尾，基本学不动）。先跑 notebook 15。")
if PUSH_FINAL_REPO == "auto":
    PUSH_FINAL_REPO = sft_repo_name(HF_USER, MODEL_NAME).replace("-s1-sft", f"-{EXP_NAME.lower()}-grpo")
from recipe_utils import persist_dir, StageCache
PERSIST = persist_dir(f"{EXP_NAME}_{MODE}", USE_DRIVE)     # 开头会弹一次 Drive 授权
CACHE = StageCache(PERSIST, enabled = RESUME)
CACHE.check_config(dict(model = MODEL_NAME, init_model = INIT_MODEL, run_sft = RUN_SFT, chat_style = CHAT_STYLE, grpo_data = GRPO_DATA,
                        grpo_filter = GRPO_FILTER, curriculum_rounds = CURRICULUM_ROUNDS, grpo_lr = GRPO_LR,
                        grpo_max_completion = GRPO_MAX_COMPLETION, epsilon_high = EPSILON_HIGH, format_weight = FORMAT_WEIGHT,
                        **{k: Q[k] for k in ("max_seq_len", "lora_rank", "grpo_steps", "grpo_prompts_per_step", "num_gen",
                                             "filter_k", "filter_max_tokens", "val_n", "eval_n", "eval_max_tokens")}))
def ADIR(name):
    return f"{PERSIST}/{name}"
print(EXP_NAME, MODE, "start from:", INIT_MODEL or MODEL_NAME, "-> push:", PUSH_FINAL_REPO, CHAT_STYLE, Q)
print("persist dir:", PERSIST, "| finished stages:", CACHE.done())

## 1. 加载模型（`fast_inference=True` = 训练和生成共用一个 vLLM 引擎）

In [ ]:
import os, torch
if torch.cuda.get_device_capability()[0] < 8:   # 再设一次，防止跳过了前面的格
    os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
from unsloth import FastLanguageModel
from recipe_utils import (RESPONSE_PARTS, STOP_STRS, TABLE_COLS, QuickEval, setup_chat, chat_prompt,
                          log_eval, print_table, sciq_mcq, sciq_format_rows, load_mcq_rows,
                          drsci_numeric_row, decontam_rows, n_chat_tokens, correctness_score, format_score,
                          soft_overlong_penalty, pass_stats, paired_delta, suggest_completion_cap, lora_delta,
                          config_kwargs, wandb_init, load_guard, restore_lora, end_runtime)

# 这一格可以反复运行：模型已经加载过就直接复用（vLLM 引擎不能在同一个进程里建第二次，以前重跑会报错）。
_KEY = (INIT_MODEL or MODEL_NAME, MAX_SEQ_LEN, LORA_RANK, GPU_UTIL, LOAD_IN_4BIT)
if load_guard(globals(), _KEY):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = INIT_MODEL or MODEL_NAME,
        max_seq_length = MAX_SEQ_LEN,
        load_in_4bit = LOAD_IN_4BIT,
        fast_inference = True,
        max_lora_rank = LORA_RANK,
        gpu_memory_utilization = GPU_UTIL,
    )
    model = FastLanguageModel.get_peft_model(
        model,
        r = LORA_RANK,
        target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha = LORA_RANK * 2,
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )
    _LOADED_KEY = _KEY
else:
    print("[i] 模型已经加载，直接复用（没有重新加载）。训练格开始时会把 LoRA 权重恢复到起点。")
# 模板 + 结束符（照 OLMo 3：最后一个 assistant 回合以 eos=<|endoftext|> 结束；generation_config 两个结束符都停）
STOP_IDS = setup_chat(tokenizer, model, style = CHAT_STYLE)
EOS_ID = tokenizer.eos_token_id
_demo = tokenizer.apply_chat_template([{"role": "user", "content": "Hi"}, {"role": "assistant", "content": "Hello"}], tokenize = False)
print(repr(_demo))
assert _demo.endswith(tokenizer.eos_token), "最后一个回合应以 eos 结束"
print("eos:", tokenizer.eos_token, EOS_ID, "| stop ids:", STOP_IDS, "| pad:", tokenizer.pad_token)

RUN_NAME = f"{EXP_NAME}-{MODEL_NAME.split('/')[-1]}-{MODE}"
wandb, REPORT_TO = wandb_init(WANDB_PROJECT, RUN_NAME, Q | dict(
    exp = EXP_NAME, mode = MODE, model = MODEL_NAME, init_model = INIT_MODEL, base_model = BASE_MODEL, chat_style = CHAT_STYLE,
    run_sft = RUN_SFT, run_grpo = RUN_GRPO, dolci = DOLCI_DATASET, dolci_include = DOLCI_INCLUDE, dolci_exclude = DOLCI_EXCLUDE,
    think_n = THINK_N, extra_sft = EXTRA_SFT, sft_select = SFT_SELECT, sft_pool_factor = SFT_POOL_FACTOR,
    coverage_norm = COVERAGE_NORM, grpo_data = GRPO_DATA, grpo_filter = GRPO_FILTER, curriculum_rounds = CURRICULUM_ROUNDS,
    grpo_lr = GRPO_LR, grpo_max_completion = GRPO_MAX_COMPLETION, epsilon_high = EPSILON_HIGH, format_weight = FORMAT_WEIGHT, overlong_cache = OVERLONG_CACHE,
    gpu = GPU_NAME, gpu_mem_gb = GPU_MEM_GB, versions = VERSIONS), WANDB_ENTITY)

## 2. GPQA 快速评测 + 诊断（只输出聚合数字；不记录题目或模型输出）

`flex` = 输出中最后一个 `(A)`–`(D)`（排行榜指标），`strict` = 第一个 `The answer is (X)`。
每题采样 `eval_samples` 次，`acc_flex` 是所有采样的平均（avg@k），`acc_flex_se` 是它的标准误（单位：分）。
**比较两个阶段时看配对差值**（第 5 节 `EVAL.compare_all()`：同一批题上的差 ± 标准误；|z| < 2 就是看不出差别）。
`gpqa_main` 包含全部 Diamond 题，所以 Main 还会报 `acc_flex_diamond_part`（其中的 Diamond 题）和 `acc_flex_rest`（其余 250 题）。

诊断指标：`repeat_rate`（某 8 词片段重复 ≥4 次，打转）、`answer_then_more`（答案之后还写了 ≥30% 的内容）、`glitch_rate`（出现 X1 里的乱码 token `\tNdrFc`）。

In [ ]:
import random, json, statistics, time
from collections import Counter
from datasets import load_dataset
from vllm import SamplingParams

EVAL = QuickEval.from_gpqa(tokenizer, Q["eval_configs"], Q["eval_n"], max_tokens = Q["eval_max_tokens"],
                           max_model_len = MAX_SEQ_LEN, samples = Q["eval_samples"], stop_ids = STOP_IDS)
RESULTS = EVAL.results

def quick_eval(tag, lora_request = None, samples = None):
    r = EVAL.run(tag, lambda texts, kw: model.fast_generate(texts, sampling_params = SamplingParams(**kw),
                                                             lora_request = lora_request), samples, cache = CACHE)
    log_eval(wandb, r, tag)

def diagnose(tag, lora_request = None, k = 3):
    # 看清楚答案后面生成了什么（SciQ 验证集，公开数据；不跳过特殊 token）
    val = load_dataset("allenai/sciq", split = "validation").select(range(k))
    prompts = [chat_prompt(tokenizer, sciq_mcq(ex, 10_000 + i)[0]) for i, ex in enumerate(val)]
    sp = SamplingParams(temperature = 0.0, max_tokens = 512, stop_token_ids = STOP_IDS, stop = STOP_STRS,
                        skip_special_tokens = False)
    for o in model.fast_generate(prompts, sampling_params = sp, lora_request = lora_request):
        c = o.outputs[0]
        print(f"--- [{tag}] finish={c.finish_reason} stop={c.stop_reason!r} tokens={len(c.token_ids)}")
        print(repr(c.text[-300:]))

def stop_check(rows, k = 16):
    # teacher forcing：模型在训练样本答案结尾预测 eos 的概率（应接近 1）；以及输出层几行的范数
    os.environ["UNSLOTH_RETURN_LOGITS"] = "1"
    im_end = tokenizer.convert_tokens_to_ids("<|im_end|>")
    glitch = tokenizer.convert_tokens_to_ids("ĉNdrFc")
    out = {}
    try:
        p_eos, p_im_end, p_glitch = [], [], []
        for r in rows[:k]:
            ids = tokenizer(tokenizer.apply_chat_template(r["messages"], tokenize = False), return_tensors = "pt",
                            add_special_tokens = False).input_ids[:, -2048:].to(model.device)
            if ids[0, -1].item() != EOS_ID:
                continue
            with torch.no_grad():
                probs = torch.softmax(model(input_ids = ids[:, :-1]).logits[0, -1].float(), -1)
            p_eos.append(probs[EOS_ID].item()); p_im_end.append(probs[im_end].item()); p_glitch.append(probs[glitch].item())
        out = dict(n = len(p_eos), p_eos_mean = round(statistics.fmean(p_eos), 4) if p_eos else None,
                   p_eos_median = round(statistics.median(p_eos), 4) if p_eos else None,
                   p_im_end_mean = round(statistics.fmean(p_im_end), 4) if p_im_end else None,
                   p_glitch_mean = round(statistics.fmean(p_glitch), 4) if p_glitch else None)
        W = model.get_output_embeddings().weight
        norms = W.float().norm(dim = -1)
        out.update(norm_eos = round(norms[EOS_ID].item(), 3), norm_im_end = round(norms[im_end].item(), 3),
                   norm_glitch = round(norms[glitch].item(), 3), norm_mean = round(norms.mean().item(), 3))
    except Exception as e:
        print("stop check skipped:", type(e).__name__, e)
    finally:
        os.environ.pop("UNSLOTH_RETURN_LOGITS", None)
    print("stop check:", out)
    return out

if INIT_MODEL:
    quick_eval("0_init")
elif RUN_BASE_EVAL:
    quick_eval("0_base", samples = 1)   # base 不收尾，每个回答都写满预算：只采样 1 次

## 3. SFT（chat 模板转成 `text` + `train_on_responses_only`）

1. **Dolci-Instruct** 按来源 / domain 过滤到科学、数学（不要逻辑题），长度 ≤ `sft_max_tokens`。
2. 可选 **Dolci-Think** 思维链（`THINK_N`）和 **额外数据源**（`EXTRA_SFT`）。
3. **覆盖度选择**（`SFT_SELECT="coverage"`）：先抽 `SFT_POOL_FACTOR` 倍候选，再按 Dr.SCI 的 EESFT 贪心挑出最能扩大 4-gram 覆盖的样本。
4. **选择题格式样本**：SciQ 训练集套 GPQA 模板，答案 = `support` 段落 + `The answer is (X).`。

全部过一遍 GPQA 13-gram 去污染。

In [ ]:
os.makedirs("data", exist_ok = True)
N_DOLCI, SFT_MAX, SHUF = Q["sft_dolci_n"], Q["sft_max_tokens"], Q["shuffle_buffer"]
POOL = SFT_POOL_FACTOR if SFT_SELECT == "coverage" else 1
inc = " ".join(DOLCI_INCLUDE); exc = " ".join(DOLCI_EXCLUDE)
inc_arg = f"--include {inc}" if inc else ""
N_POOL = N_DOLCI * POOL
if RUN_SFT:
    !python scripts/prepare_dolci_sft.py --dataset {DOLCI_DATASET} --out data/dolci.jsonl --n {N_POOL} --tokenizer {MODEL_NAME} --max-tokens {SFT_MAX} --shuffle-buffer {SHUF} {inc_arg} --exclude {exc}
    if THINK_N > 0:
        t_inc = " ".join(THINK_INCLUDE); N_THINK_POOL = THINK_N * POOL
        !python scripts/prepare_dolci_sft.py --dataset {THINK_DATASET} --out data/think.jsonl --n {N_THINK_POOL} --tokenizer {MODEL_NAME} --max-tokens {SFT_MAX} --shuffle-buffer {SHUF} --include {t_inc} --exclude {exc}

In [ ]:
DATA = {}
if RUN_SFT:
    from recipe_utils import prepare_sft_sources
    from select_coverage import select as coverage_select, hashed_coverage

    def _load(path):
        return [json.loads(l) for l in open(path)] if os.path.exists(path) else []

    pool = [{"messages": r["prompt"] + r["completion"], "source": r["source"], "n_tokens": r["n_tokens"], "part": "dolci"}
            for r in _load("data/dolci.jsonl")]
    pool += [{"messages": r["prompt"] + r["completion"], "source": r["source"], "n_tokens": r["n_tokens"], "part": "think"}
             for r in (_load("data/think.jsonl") if THINK_N > 0 else [])]
    if EXTRA_SFT:
        pool += [dict(r, part = "extra") for r in prepare_sft_sources(EXTRA_SFT, MODEL_NAME, SFT_MAX, scale = POOL, shuffle_buffer = SHUF)]
    pool, dropped = decontam_rows(pool, lambda r: {"messages": r["messages"]}, "sft_pool")
    target = N_DOLCI + (THINK_N if THINK_N > 0 else 0) + sum(s["n"] for s in EXTRA_SFT)
    texts = ["\n".join(m["content"] for m in r["messages"]) for r in pool]
    if SFT_SELECT == "coverage" and len(pool) > target:
        t0 = time.time()
        idx = coverage_select(texts, target, norm = COVERAGE_NORM)
        rand = random.Random(0).sample(range(len(pool)), min(target, len(pool)))
        cov = dict(selected_4grams = hashed_coverage([texts[i] for i in idx]),
                   random_4grams = hashed_coverage([texts[i] for i in rand]), minutes = round((time.time() - t0) / 60, 1))
        print("coverage selection:", cov)
    else:
        idx, cov = list(range(min(target, len(pool)))), {}
    chosen = [pool[i] for i in idx]
    sft_mcq = sciq_format_rows(Q["sft_mcq_n"])
    sft_rows = [{"messages": r["messages"]} for r in chosen] + sft_mcq
    random.Random(0).shuffle(sft_rows)
    DATA = dict(pool = len(pool), selected = len(chosen), sciq_format = len(sft_mcq), dropped_decontam = dropped,
                select = SFT_SELECT, coverage = cov, tokens = sum(r["n_tokens"] or 0 for r in chosen),
                by_part = Counter(r["part"] for r in chosen).most_common(),
                sources = Counter(r["source"] for r in chosen).most_common(10))
    wandb.config.update({"data": DATA}, allow_val_change = True)
    print(DATA)

In [ ]:
SFT, STOP = {}, {}
if RUN_SFT:
    from datasets import Dataset
    from trl import SFTTrainer, SFTConfig
    from unsloth.chat_templates import train_on_responses_only

    # 文本以 eos 结束（模板最后一个回合写的就是 eos），TRL 不会再额外追加
    sft_ds = Dataset.from_list([{"text": tokenizer.apply_chat_template(r["messages"], tokenize = False)} for r in sft_rows])
    trainer = SFTTrainer(
        model = model,
        tokenizer = tokenizer,
        train_dataset = sft_ds,
        args = SFTConfig(
            dataset_text_field = "text",
            max_length = MAX_SEQ_LEN,
            per_device_train_batch_size = Q["sft_batch"],
            gradient_accumulation_steps = Q["sft_accum"],
            warmup_steps = 10,
            max_steps = Q["sft_steps"],
            num_train_epochs = 1,
            learning_rate = Q["sft_lr"],
            logging_steps = 5,
            optim = "adamw_8bit",
            weight_decay = 0.001,
            lr_scheduler_type = "linear",
            seed = 3407,
            report_to = REPORT_TO,
            run_name = RUN_NAME + "-sft",
            output_dir = "outputs/sft",
            save_strategy = "no",
        ),
    )
    instr_part, resp_part = RESPONSE_PARTS[CHAT_STYLE]
    trainer = train_on_responses_only(trainer, instruction_part = instr_part, response_part = resp_part)
    # 诊断：训练标签的最后一个 token 应该是 eos（模型要学会写它）
    try:
        _labels = [t for t in trainer.train_dataset[0]["labels"] if t != -100]
        print("label tail:", repr(tokenizer.decode(_labels[-12:])), "| ends with eos:", _labels[-1] == EOS_ID)
    except Exception as e:
        print("label check skipped:", e)

    print("fresh LoRA:", restore_lora(model))     # 重跑这一格时从头开始
    sft_stats = trainer.train()
    sft_losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
    SFT = dict(steps = trainer.state.global_step, minutes = round(sft_stats.metrics["train_runtime"] / 60, 1),
               loss_first = sft_losses[0] if sft_losses else None, loss_last = sft_losses[-1] if sft_losses else None)
    print(SFT)
    STOP = stop_check(sft_rows)
    model.save_lora(ADIR("sft_lora"))
    quick_eval("1_sft", model.load_lora(ADIR("sft_lora")))
    diagnose("1_sft", model.load_lora(ADIR("sft_lora")))
    if PUSH_SFT_REPO:
        try:
            model.push_to_hub_merged(PUSH_SFT_REPO, tokenizer, save_method = "merged_16bit",
                                     private = True, token = os.environ.get("HF_TOKEN"))
        except TypeError:   # 旧版 Unsloth 没有 private 参数：先在 HF 网页上建好私有仓库再推
            model.push_to_hub_merged(PUSH_SFT_REPO, tokenizer, save_method = "merged_16bit",
                                     token = os.environ.get("HF_TOKEN"))
        print("SFT pushed to", PUSH_SFT_REPO, "→ notebook 12 的 INIT_MODEL 填它")
    del sft_ds, trainer
    torch.cuda.empty_cache(); import gc; gc.collect()

## 4. GRPO（`vllm_sampling_params` + 多个奖励函数）

**题库**（`GRPO_DATA`）：
- `science`（默认）：50% **MegaScience 留出的 4 选 1 科学题**（NVIDIA OpenScience 的研究生水平选择题，经 MegaScience 的 LLM 去污染和难度筛选；
  和 GPQA 同一种题面；notebook 15 的 SFT 只用了另外 55% 的题，这里的题没见过解答）+ 20% MMLU-Pro STEM + 30% Dr.SCI 数值题。
- `mix`（X1b 用的）：一半 **MMLU-Pro STEM**（物理/化学/生物/工程/数学，抽成 4 选 1，套 GPQA 模板，答案看最后一个 `(A)`–`(D)`），
  一半 **Dr.SCI 数值题**（[MiniByte-666/Dr.SCI](https://huggingface.co/datasets/MiniByte-666/Dr.SCI) 的可验证题里参考答案是数字的；用它自带的 `\boxed{}` 提示，相对误差 ≤1% 算对；没有 25% 的猜中率）。
- 课程（`CURRICULUM_ROUNDS=2`）：第一段 Dr.SCI 用 Qwen3-32B 通过率 0.5–0.875 的题，第二段用 0.25–0.375 的题；每段开始前用**当前模型**每题采样 4 次，只留答对 1–3 次的（`GRPO_FILTER`）。

**奖励**：
- `correctness_reward`：答对 +2，答错 0，找不到答案 −1
- `format_reward`：答案以规定形式恰好出现一次 +0.5（选择题 `The answer is (X)`，数值题 `\boxed{}`）
- `overlong_reward`：DAPO 软超长惩罚——长度进入上限前的最后一段后线性扣到 −1，被截断 −1

**怎么判断 GRPO 有没有用**（X1b 的教训：别只看 GPQA 上一次采样的分数）：
1. **留出验证集** `rl_val`：和训练题同分布、但不参与训练的 `val_n` 道题，每题采样 4 次（温度 1.0，和 rollouts 一样）。训练前、每段之后各测一次，
   并给出配对差值 ± 标准误。**它不涨 = GRPO 没学到能泛化的东西**（步数 / 学习率 / 题量的问题）；它涨了但 GPQA 不涨 = 迁移的问题（题库和 GPQA 差太远）。
2. `adapter_change`：这一段训练让 LoRA 权重相对上一段变了多少（相对范数）。X1b 的学习率 5e-6 × 100 步，每个参数最多移动 5e-4。
3. W&B 上的 `rewards/correctness_reward/mean`：每步是不同的题，单步噪声很大，要看 20 步以上的平均。

**每步的规模**：`grpo_prompts_per_step` 道题 × `num_gen` 个回答。X1b 是 4 × 8（梯度噪声大，100 步只见 400 道题）；现在 a100 是 16 × 8，120 步见 1920 道题。
生成很便宜（vLLM 并行），每步的固定开销（合并 / 还原权重、暂停引擎）占大头，所以加大每步题数比加步数划算。

In [ ]:
import numpy as np
from datasets import Dataset

STEPS_PER_STAGE = max(1, Q["grpo_steps"] // max(1, CURRICULUM_ROUNDS))
NEED_PER_STAGE = STEPS_PER_STAGE * Q["grpo_prompts_per_step"]
CAND_PER_STAGE = int(NEED_PER_STAGE * (1.7 if GRPO_FILTER else 1))   # X1b：预筛保留约 55%；不够的用没筛过的题补
VAL_N = Q["val_n"]

def _mcq_rows(source, n, skip = 0):
    return [{"prompt": [{"role": "user", "content": r["prompt"]}], "answer": r["answer"], "answer_type": "mcq",
             "source": r["source"]} for r in load_mcq_rows(source, n, skip = skip)]

# MegaScience 留出的 4 选 1 科学题（GPQA 题面）。和 notebook 15 用同一个脚本、按题目哈希拆分，所以这里的题 SFT 没见过解答。
SCI_POOL, SCI_VAL = [], []
if RUN_GRPO and GRPO_DATA == "science":
    from science_mix import load_mcq_pool
    from recipe_utils import run_script
    if not os.path.exists("data/s1/rl_mcq.jsonl"):
        run_script(["scripts/science_mix.py", "--out-dir", "data/s1", "--pools-only"], show_tail = 40)   # 下载 MegaScience（约 1.9 GB），约 3 分钟
    SCI_POOL, SCI_VAL = load_mcq_pool("data/s1/rl_mcq.jsonl"), load_mcq_pool("data/s1/val_mcq.jsonl")
    print("science MCQ pool:", len(SCI_POOL), "| validation:", len(SCI_VAL))

DRSCI_RANGES = [(0.5, 0.875), (0.25, 0.375)] if CURRICULUM_ROUNDS > 1 else [(0.25, 0.875)]
_drsci_cache = {}
def _drsci_rows(n, rng, start_frac = 0.0):
    if "ds" not in _drsci_cache:
        _drsci_cache["ds"] = load_dataset("MiniByte-666/Dr.SCI", data_files = "Dr_SCI_verifiable.parquet",
                                          split = "train").shuffle(seed = 0)
    ds, out = _drsci_cache["ds"], []
    for row in ds.select(range(int(start_frac * len(ds)), len(ds))):
        r = drsci_numeric_row(row, subjects = DRSCI_SUBJECTS, difficulty = rng)
        if r:
            out.append(r)
            if len(out) >= n:
                break
    return out

def _fits(r):
    return n_chat_tokens(tokenizer, r["prompt"], add_generation_prompt = True) <= MAX_PROMPT_LEN

# 题目长度上限（TRL 不支持 max_prompt_length，像 Unsloth 示例那样直接过滤）
MAX_PROMPT_LEN = 1024
PRESET_CAP = min(Q["grpo_max_completion"], MAX_SEQ_LEN - MAX_PROMPT_LEN)
CAP = {"max": PRESET_CAP, "cache": min(OVERLONG_CACHE, PRESET_CAP // 4)}   # 每段开始前按模型自己的长度更新

# 留出验证集：和训练题同分布、不参与训练（MMLU-Pro 取打乱后靠后的部分，Dr.SCI 取后 25%）
def val_pool(n):
    if n <= 0:
        return []
    if GRPO_DATA == "science":
        rows = (SCI_VAL[: n // 2] + _mcq_rows("mmlu_pro_stem", n // 5, skip = 10_000)
                + _drsci_rows(n - n // 2 - n // 5, (0.25, 0.875), start_frac = 0.75))
    elif GRPO_DATA == "sciq":
        rows = _mcq_rows("sciq", n, skip = 11_000)
    elif GRPO_DATA == "drsci":
        rows = _drsci_rows(n, (0.25, 0.875), start_frac = 0.75)
    elif GRPO_DATA == "mix":
        rows = _mcq_rows("mmlu_pro_stem", n // 2, skip = 10_000) + _drsci_rows(n - n // 2, (0.25, 0.875), start_frac = 0.75)
    else:
        rows = _mcq_rows("mmlu_pro_stem", n, skip = 10_000)
    rows, _ = decontam_rows(rows, lambda r: {"prompt": r["prompt"]}, "grpo_val")
    return [r for r in rows if _fits(r)]

VAL_ROWS = val_pool(VAL_N) if RUN_GRPO else []
_VAL_PROMPTS = {r["prompt"][0]["content"] for r in VAL_ROWS}
RL_VAL, _RL_VAL_PQ = {}, {}

def _sample(rows, lora_request, k, max_tokens, seed):
    sp = SamplingParams(n = k, temperature = 1.0, top_p = 1.0, max_tokens = max_tokens, seed = seed,
                        stop_token_ids = STOP_IDS, stop = STOP_STRS)
    outs = model.fast_generate([chat_prompt(tokenizer, r["prompt"][0]["content"]) for r in rows],
                               sampling_params = sp, lora_request = lora_request)
    return pass_stats(outs, rows)

def rl_val(tag, lora_request = None):
    # 留出题上的通过率（答对且正常收尾）；和上一次测量做配对比较
    if not VAL_ROWS:
        return {}
    if CACHE.has(f"rl_val_{tag}") and CACHE.load(f"rl_val_{tag}")["prompts"] == len(VAL_ROWS):
        _saved = CACHE.load(f"rl_val_{tag}")                 # 断线后续跑：直接读回
        RL_VAL[tag], _RL_VAL_PQ[tag] = _saved["st"], _saved["pq"]
        print("rl_val", tag, "(cached)", _saved["st"])
        return _saved["st"]
    t0 = time.time()
    st = _sample(VAL_ROWS, lora_request, 4, Q["filter_max_tokens"], seed = 11)
    pq = [c / st["samples"] for c in st.pop("correct_counts")]
    st.pop("stopped_lengths")
    st["minutes"] = round((time.time() - t0) / 60, 1)
    if _RL_VAL_PQ:
        prev = list(_RL_VAL_PQ)[-1]
        st["vs_" + prev] = paired_delta(_RL_VAL_PQ[prev], pq)
        first = list(_RL_VAL_PQ)[0]
        if first != prev:
            st["vs_" + first] = paired_delta(_RL_VAL_PQ[first], pq)
    RL_VAL[tag], _RL_VAL_PQ[tag] = st, pq
    CACHE.save(f"rl_val_{tag}", {"st": st, "pq": pq, "prompts": len(VAL_ROWS)})
    print("rl_val", tag, st)
    if wandb.run is not None:
        wandb.run.summary[f"rl_val/{tag}/pass_rate"] = st["pass_rate"]
    return st

def stage_pool(stage):
    n = CAND_PER_STAGE
    if GRPO_DATA == "science":
        n_sci, n_pro = n // 2, n // 5
        rows = (SCI_POOL[stage * n_sci:(stage + 1) * n_sci] + _mcq_rows("mmlu_pro_stem", n_pro * (stage + 1))[-n_pro:]
                + _drsci_rows(n - n_sci - n_pro, DRSCI_RANGES[min(stage, len(DRSCI_RANGES) - 1)]))
    elif GRPO_DATA == "mix":
        rows = _mcq_rows("mmlu_pro_stem", n // 2 * (stage + 1))[-(n // 2):] + _drsci_rows(n - n // 2, DRSCI_RANGES[min(stage, len(DRSCI_RANGES) - 1)])
    elif GRPO_DATA == "drsci":
        rows = _drsci_rows(n, DRSCI_RANGES[min(stage, len(DRSCI_RANGES) - 1)])
    elif GRPO_DATA == "sciq":
        rows = _mcq_rows("sciq", n * (stage + 1), skip = 4000)[-n:]      # SciQ 前段用于 SFT 格式样本
    else:
        rows = _mcq_rows("mmlu_pro_stem", n * (stage + 1))[-n:]
    rows = [r for r in rows if r["prompt"][0]["content"] not in _VAL_PROMPTS]   # 不和验证集重复
    rows, dropped = decontam_rows(rows, lambda r: {"prompt": r["prompt"]}, f"grpo_pool_s{stage}")
    random.Random(stage).shuffle(rows)
    return rows

def pass_filter(rows, lora_request, need):
    # 当前模型每题采样 k 次，只留答对 1..k-1 次的题（全对 / 全错的组没有学习信号）
    k = Q["filter_k"]
    t0 = time.time()
    st = _sample(rows, lora_request, k, Q["filter_max_tokens"], seed = 7)
    counts, lengths = st.pop("correct_counts"), st.pop("stopped_lengths")
    keep = [r for r, c in zip(rows, counts) if 0 < c < k]
    info = dict(st, candidates = len(rows), kept = len(keep), minutes = round((time.time() - t0) / 60, 1))
    print("pass filter:", info)
    if len(keep) < need:
        print(f"[!] only {len(keep)} prompts passed the filter (need {need}); topping up with unfiltered ones")
        kept_ids = {id(r) for r in keep}
        keep += [r for r in rows if id(r) not in kept_ids][: need - len(keep)]
    return keep[:need], info, lengths

def format_reward(completions, answer_type, **kwargs):
    return [format_score(c[0]["content"], t, FORMAT_WEIGHT) for c, t in zip(completions, answer_type)]

def correctness_reward(completions, answer, answer_type, **kwargs):
    return [correctness_score(c[0]["content"], a, t) for c, a, t in zip(completions, answer, answer_type)]

def overlong_reward(completions, completion_ids = None, **kwargs):
    if completion_ids is None:
        completion_ids = [tokenizer(c[0]["content"], add_special_tokens = False)["input_ids"] for c in completions]
    return [soft_overlong_penalty(len(ids), CAP["max"], CAP["cache"]) for ids in completion_ids]

print(dict(stages = CURRICULUM_ROUNDS, steps_per_stage = STEPS_PER_STAGE, prompts_per_step = Q["grpo_prompts_per_step"],
           prompts_per_stage = NEED_PER_STAGE, candidates_per_stage = CAND_PER_STAGE, rollouts_total =
           Q["grpo_steps"] * Q["grpo_prompts_per_step"] * Q["num_gen"], preset_cap = PRESET_CAP, val_prompts = len(VAL_ROWS)))

In [ ]:
GRPO = {}
if RUN_GRPO:
    from trl import GRPOConfig, GRPOTrainer
    vllm_sampling_params = SamplingParams(min_p = 0.1, top_p = 1.0, top_k = -1, seed = 3407,
                                          stop_token_ids = STOP_IDS, stop = STOP_STRS)
    NUM_GEN = Q["num_gen"]
    # 每次运行这一格都从起点开始：RUN_SFT 时是 SFT 的 adapter，否则是全新的 LoRA（= INIT_MODEL 本身）。
    # 已经完成的阶段（RESUME=True）不重训，直接从持久目录读回 adapter 和结果，所以断线后 Run all 会接着跑。
    prev_dir = ADIR("sft_lora") if RUN_SFT else None
    print("start adapter:", restore_lora(model, prev_dir))
    lora_req = model.load_lora(prev_dir) if RUN_SFT else None
    stages = []
    RL_VAL.clear(); _RL_VAL_PQ.clear()
    rl_val("before_grpo", lora_req)

    def _avg(log, key, part):
        vals = [h[key] for h in log if key in h]
        k = max(1, len(vals) // 5)
        return round(float(np.mean(vals[:k] if part == "first" else vals[-k:])), 3) if vals else None

    for stage in range(CURRICULUM_ROUNDS):
        cur_dir = ADIR(f"grpo_lora_s{stage}")
        if CACHE.has(f"grpo_s{stage}") and os.path.exists(f"{cur_dir}/adapter_model.safetensors"):
            s = CACHE.load(f"grpo_s{stage}")
            print(f"[i] 第 {stage} 段已经完成，读回 adapter：", restore_lora(model, cur_dir))
            lora_req = model.load_lora(cur_dir)
            s["rl_val"] = rl_val(f"after_s{stage}", lora_req)
            stages.append(s); prev_dir = cur_dir
            continue
        rows = [r for r in stage_pool(stage) if _fits(r)]
        filt, lengths = {}, []
        if GRPO_FILTER:
            rows, filt, lengths = pass_filter(rows, lora_req, NEED_PER_STAGE)
        else:
            rows = rows[:NEED_PER_STAGE]
        # 回答上限：2 × 模型自己收尾长度的 99 分位（停不下来的回答会让整步等它写满上限）
        CAP["max"] = (suggest_completion_cap(lengths, PRESET_CAP) if GRPO_MAX_COMPLETION == "auto"
                      else min(int(GRPO_MAX_COMPLETION), MAX_SEQ_LEN - MAX_PROMPT_LEN))
        CAP["cache"] = min(OVERLONG_CACHE, CAP["max"] // 4)
        steps = max(1, min(STEPS_PER_STAGE, len(rows) // Q["grpo_prompts_per_step"]))
        print(dict(stage = stage, prompts = len(rows), steps = steps, max_completion = CAP["max"], overlong_cache = CAP["cache"]))
        ds = Dataset.from_list([{k: r[k] for k in ("prompt", "answer", "answer_type")} for r in rows])
        args = GRPOConfig(**config_kwargs(
            GRPOConfig,
            vllm_sampling_params = vllm_sampling_params,
            generation_kwargs = {"stop_token_ids": STOP_IDS},
            temperature = 1.0,
            learning_rate = GRPO_LR,
            weight_decay = 0.001,
            warmup_steps = max(1, steps // 10) if stage == 0 else 0,
            lr_scheduler_type = "constant_with_warmup",      # OLMo 3 RL：常数学习率
            optim = "adamw_8bit",
            logging_steps = 1,
            per_device_train_batch_size = NUM_GEN,            # 一个 micro-batch = 一道题的 NUM_GEN 个回答
            gradient_accumulation_steps = Q["grpo_prompts_per_step"],   # 每步 = 这么多道题（一次生成，累积梯度后更新一次）
            num_generations = NUM_GEN,
            max_completion_length = CAP["max"],
            epsilon_high = EPSILON_HIGH,                      # clip-higher（DAPO / OLMo 3）；loss_type/beta 用 Unsloth 默认 dapo、0
            max_steps = steps,
            save_steps = 10_000,
            log_completions = False,                          # 不往 W&B 上传生成内容
            report_to = REPORT_TO,
            run_name = f"{RUN_NAME}-grpo-s{stage}",
            output_dir = f"outputs/grpo_s{stage}",
        ))
        trainer = GRPOTrainer(model = model, processing_class = tokenizer,
                              reward_funcs = [format_reward, correctness_reward, overlong_reward],
                              args = args, train_dataset = ds)
        stats = trainer.train()
        log = trainer.state.log_history
        s = dict(stage = stage, steps = trainer.state.global_step, minutes = round(stats.metrics["train_runtime"] / 60, 1),
                 prompts = len(ds), by_type = dict(Counter(r["answer_type"] for r in rows)), filter = filt,
                 max_completion = CAP["max"])
        for key in ["reward", "rewards/format_reward/mean", "rewards/correctness_reward/mean", "rewards/overlong_reward/mean",
                    "completions/mean_length", "completions/clipped_ratio", "frac_reward_zero_std"]:
            if any(key in h for h in log):
                s[key] = {"first": _avg(log, key, "first"), "last": _avg(log, key, "last")}
        model.save_lora(cur_dir)                               # 直接存在持久目录
        lora_req = model.load_lora(cur_dir)
        if prev_dir:
            try:
                s["adapter_change"] = lora_delta(prev_dir, cur_dir)   # 这一段让 LoRA 权重变了多少
            except Exception as e:
                print("adapter change skipped:", type(e).__name__, e)
        prev_dir = cur_dir
        del trainer, ds
        torch.cuda.empty_cache()
        CACHE.save(f"grpo_s{stage}", s)                        # 这一段完成：断线后不会重训
        s["rl_val"] = rl_val(f"after_s{stage}", lora_req)
        stages.append(s)
        print(s)
    GRPO = dict(stages = stages, grpo_data = GRPO_DATA, prompts_per_step = Q["grpo_prompts_per_step"], lr = GRPO_LR,
                rl_val = RL_VAL)
    quick_eval("2_grpo", lora_req)
    diagnose("2_grpo", lora_req)

## 5. 汇总（W&B 上也有同样的表；最后打印的 JSON 可以贴回给 Claude）

In [ ]:
rows = print_table(RESULTS)
print("\n阶段之间的配对比较（同一批题，后一阶段 − 前一阶段，单位：分）：")
COMPARE = EVAL.compare_all()
if RL_VAL:
    print("\n留出验证集通过率（GRPO 学到东西没有）：", {k: (v.get("pass_rate"), v.get("pass_by_type")) for k, v in RL_VAL.items()})
print("SFT:", SFT); print("STOP:", STOP); print("GRPO:", GRPO)
REPORT = dict(notebook = "11_sft_grpo_unsloth", exp = EXP_NAME, mode = MODE, model = MODEL_NAME, init_model = INIT_MODEL,
              chat_style = CHAT_STYLE, gpu = GPU_NAME, gpu_mem_gb = GPU_MEM_GB, q = Q, dolci = DOLCI_DATASET,
              dolci_include = DOLCI_INCLUDE, think_n = THINK_N, extra_sft = EXTRA_SFT, sft_select = SFT_SELECT,
              grpo_data = GRPO_DATA, grpo_filter = GRPO_FILTER, curriculum_rounds = CURRICULUM_ROUNDS, grpo_lr = GRPO_LR,
              epsilon_high = EPSILON_HIGH, data = DATA, sft = SFT, stop_check = STOP, grpo = GRPO, quick_eval = RESULTS,
              compare = COMPARE, versions = VERSIONS, wandb = wandb.run.url if REPORT_TO == "wandb" else None)
os.makedirs("outputs", exist_ok = True)
json.dump(REPORT, open(f"outputs/report_{EXP_NAME}.json", "w"), indent = 1, default = str)
if REPORT_TO == "wandb":
    wandb.log({"gpqa_quick_eval": wandb.Table(columns = TABLE_COLS, data = rows)})
    art = wandb.Artifact(f"report-{RUN_NAME}", type = "report")
    art.add_file(f"outputs/report_{EXP_NAME}.json")
    wandb.log_artifact(art)
    wandb.finish()
json.dump(REPORT, open(f"{PERSIST}/report_{EXP_NAME}.json", "w"), indent = 1, default = str)   # 断开之后也能在 Drive 里找到
print("=" * 30, "把下面整段 JSON 贴回给 Claude（也存在 Drive：", f"{PERSIST}/report_{EXP_NAME}.json", "）", "=" * 30)
print(json.dumps(REPORT, ensure_ascii = False, default = str))

## 6. 合并保存 → 推到 Hub → 用 `main` 分支的 `01` 做正式评测

默认合并最后一个阶段并推到 HF 私有仓库（`PUSH_FINAL_REPO="auto"`）；推送失败就存到持久目录。第 5 节的配对比较如果显示最后一段没有提升，
把 `FINAL_ADAPTER` 设成 `"grpo_lora_s0"` 再跑这一格；如果 GRPO 整体没有提升，直接用起点（S1 / DPO 模型）去做正式评测，不用再推。

`merged_16bit` 把 LoRA 合并进权重（排行榜要求）；保存的 tokenizer 带着本 notebook 的模板和 eos=`<|endoftext|>`，`generation_config` 里两个结束符都在，锁定评测会正确停止。
推到 Hub 后**换一个新的 runtime** 打开 [`01_eval_gpqa.ipynb`](https://colab.research.google.com/github/yy726/GPQA-eval/blob/main/notebooks/01_eval_gpqa.ipynb)（生成预算 8192）。A100 上训练的模型提交时在 `notes` 里注明 A100（不参与排名）。

In [ ]:
if FINAL_ADAPTER:
    # 改为合并较早的阶段（例如第 5 节的配对比较显示 GRPO 没有提升时）
    print("merging adapter:", FINAL_ADAPTER, restore_lora(model, ADIR(FINAL_ADAPTER)))
_final_key = f"pushed_{FINAL_ADAPTER or 'last'}"
if CACHE.has(_final_key):
    print("[i] 已经推送 / 保存过：", CACHE.load(_final_key))
else:
    _where = PUSH_FINAL_REPO
    try:
        if not PUSH_FINAL_REPO:
            raise ValueError("PUSH_FINAL_REPO is None")
        try:
            model.push_to_hub_merged(PUSH_FINAL_REPO, tokenizer, save_method = "merged_16bit", private = True,
                                     token = os.environ.get("HF_TOKEN"))
        except TypeError:   # 旧版 Unsloth 没有 private 参数
            model.push_to_hub_merged(PUSH_FINAL_REPO, tokenizer, save_method = "merged_16bit", token = os.environ.get("HF_TOKEN"))
        print(f"pushed (private): https://huggingface.co/{PUSH_FINAL_REPO}（正式评测前改成公开）")
    except Exception as e:
        _where = ADIR("merged_16bit")
        print("[!] 没有推到 HF：", type(e).__name__, str(e)[:200], "→ 合并模型存到", _where)
        model.save_pretrained_merged(_where, tokenizer, save_method = "merged_16bit")
    CACHE.save(_final_key, {"model": _where})

## 7. 断开 runtime

adapter、合并模型、报告都已经在 Google Drive / HF 上。`AUTO_DISCONNECT=True` 时这一格在 60 秒后释放 runtime，不再消耗 CU——
所以可以 **Run all 之后离开**：中途断线就重新打开、再 Run all，已经完成的评测和 GRPO 阶段会直接跳过。

In [ ]:
end_runtime(AUTO_DISCONNECT)